# Exportação da camada Gold para CSV

Este notebook exporta as sete tabelas Gold do catálogo para CSV, com cabeçalho e separador ponto e vírgula.

Os arquivos são gravados no Unity Catalog Volume em uma pasta por tabela.
Uma execução nova sobrescreve a exportação anterior de cada tabela.

A tabela `visao_360_uc` pode conter dados pessoais, como CPF/CNPJ, nome e endereço.
O Volume de destino deve ter acesso restrito às pessoas e serviços autorizados.

In [0]:
# Definindo o catálogo e schema do ambiente
CATALOG = "workspace"
SCHEMA_GOLD = "gold"

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.gold.gold_export_volume;

In [0]:
CATALOG = "workspace"
SCHEMA_GOLD = "gold"

PATH_EXPORT = (
    "/Volumes/workspace/gold/gold_export_volume/export_csv"
)

TABELAS_GOLD = [
    "diagnostico_receita",
    "indicadores_dec_fec",
    "fila_inspecao",
    "visao_360_uc",
    "consumo_mensal_uc",
    "consumo_receita_bairro",
    "variacao_consumo_uc"
]

print(f"Iniciando exportação de {len(TABELAS_GOLD)} tabelas Gold.")
print(f"Destino: {PATH_EXPORT}")

# Verifica todas as tabelas antes de começar, para não terminar
# com uma exportação parcial por nome incorreto ou tabela ausente.
tabelas_ausentes = [
    nome
    for nome in TABELAS_GOLD
    if not spark.catalog.tableExists(
        f"{CATALOG}.{SCHEMA_GOLD}.{nome}"
    )
]

if tabelas_ausentes:
    raise ValueError(
        f"Tabelas Gold não encontradas: {tabelas_ausentes}"
    )

for nome_tabela in TABELAS_GOLD:
    tabela_completa = f"{CATALOG}.{SCHEMA_GOLD}.{nome_tabela}"
    caminho_saida = f"{PATH_EXPORT}/{nome_tabela}"

    df = spark.table(tabela_completa)

    (
        df.write
        .format("csv")
        .mode("overwrite")
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", "UTF-8")
        .option("quoteAll", "true")
        .save(caminho_saida)
    )

    print(f"[OK] {tabela_completa} -> {caminho_saida}")

print("Exportação das sete tabelas Gold concluída.")